# PDU Exam Observer — self-contained ST-GCN training

Two explicit modes are available. `SYNTHETIC_SMOKE` proves the CPU/GPU training → ONNX → bundle path and creates a `DEMO_ONLY` model. `RESEARCH` requires an allowlisted pose-only export plus a frozen protocol record. Raw video must never be uploaded. Outputs are evidence for human review, never automatic discipline or proof of research performance.


In [ ]:
from pathlib import Path

PINNED_REQUIREMENTS = r'''
torch==2.13.0
onnx==1.22.0
onnxruntime==1.29.0
onnxscript==0.7.1
numpy==2.4.6
scikit-learn==1.9.0
matplotlib==3.11.1

'''
Path('/content/pdu-requirements.txt').write_text(PINNED_REQUIREMENTS)
%pip install --quiet -r /content/pdu-requirements.txt


In [ ]:
import base64
import io
import sys
import zipfile

PIPELINE_ARCHIVE_B64 = (
    '...'
)
WORK_ROOT = '/content/pdu-training'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PIPELINE_ARCHIVE_B64))) as zf:
    zf.extractall(WORK_ROOT)
sys.path.insert(0, WORK_ROOT)
sys.path.insert(0, WORK_ROOT + '/src')
print('Embedded source restored; no repository or remote code fetched.')


In [ ]:
MODE = 'SYNTHETIC_SMOKE'  # or 'RESEARCH'
OUTPUT_ROOT = '/content/pdu-training-output'
RESEARCH_EXPORT = None
PROTOCOL_FREEZE = None
if MODE not in {'SYNTHETIC_SMOKE', 'RESEARCH'}:
    raise ValueError('MODE must be SYNTHETIC_SMOKE or RESEARCH')
if MODE == 'RESEARCH' and (not RESEARCH_EXPORT or not PROTOCOL_FREEZE):
    raise RuntimeError(
        'RESEARCH_EXPORT_REQUIRED: pose-only ZIP and '
        'frozen protocol JSON required'
    )


In [ ]:
from pathlib import Path

from research.training.showcase.v3.pipeline import (
    run_research_training,
    run_synthetic_smoke,
)

if MODE == 'SYNTHETIC_SMOKE':
    delivery = run_synthetic_smoke(Path(OUTPUT_ROOT), epochs=1)
else:
    delivery = run_research_training(
        Path(RESEARCH_EXPORT), Path(PROTOCOL_FREEZE), Path(OUTPUT_ROOT)
    )
print({
    'model_zip': str(delivery.model_zip),
    'reports_zip': str(delivery.reports_zip),
})


In [ ]:
from google.colab import files

files.download(str(delivery.model_zip))
files.download(str(delivery.reports_zip))
